# Adrienne Rich — Workshop Notebook
**CompLit 126x — Love in Context**

This notebook implements the prompt chain designed by the Adrienne Rich workshop group. Their chain focused on the *Twenty-One Love Poems* sequence and used an **iterate → compare → feed actual poems → cross-chat critique** architecture.

```
basic command     ──▶  contextualize +    ──▶  iterate ×3
prompt                 catalog (21 Love        times
                       Poems)                    │
                                                 ▼
                                          identify shortfalls
                                          + compare within
                                          context
                                                 │
                                                 ▼
feed in actual   ──▶  generate improved   ──▶  new chat
poem                  poem ★                   (critique)
                                                 │
                                                 ▼
                                          final generation
```

**What makes this chain interesting:** The group discovered that Claude could *identify* Rich's themes accurately but failed to *apply* them. So they built a loop: generate, compare, identify shortfalls — then fed in the actual poems as ground truth before generating the final version. The cross-chat critique step (moving the poem to a fresh context) simulates a second reader.

---
**Run the cells in order.** Each step builds on the previous one.

## Setup
Run the two cells below once at the start of your session.

In [ ]:
# Install the OpenAI SDK (run once per session)
%pip install openai --quiet
print("✓ Installed")

In [ ]:
from openai import OpenAI
import json
import os

# ── API Key ──────────────────────────────────────────────────────────────────
# In Google Colab:
#   1. Click the 🔑 (Secrets) icon in the left sidebar
#   2. Add a secret named  OPENAI_API_KEY  with your key
#   3. Toggle "Notebook access" to ON, then run this cell
#
# Locally: set the OPENAI_API_KEY environment variable

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
    print("✓ Using Colab Secrets")
except (ImportError, Exception):
    api_key = os.environ.get('OPENAI_API_KEY')
    print("✓ Using environment variable")

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o"

# Helper: call the model and return the text
def ask(prompt, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.responses.create(model=MODEL, input=messages)
    return response.output_text

print(f"✓ Client ready | Model: {MODEL}")

---
## Step 1: One-Shot (Basic Command Prompt)

The group started with a deliberately vague prompt — just asking for an Adrienne Rich poem without specifying the collection. This establishes a baseline: what does the model produce when given minimal context?

In [ ]:
# ── Step 1: One-shot ─────────────────────────────────────────────────────────
# Deliberately vague — no collection, no constraints.

one_shot = ask(
    "Generate a poem that Adrienne Rich would write."
)

print("ONE-SHOT RESULT")
print("═" * 60)
print(one_shot)
print("\n" + "═" * 60)
print("\n↓ Look at this. What does it get right? What's generic?")
print("  The group found: themes were identified, but not applied.")

---
## Step 2: Contextualize + Catalog

Now we narrow the context: specify the *Twenty-One Love Poems* collection, and ask the model to generate **three** different poems in that style. This gives us material to compare.

In [ ]:
# ── Step 2: Contextualize and iterate ×3 ──────────────────────────────────────
# Specify the collection. Ask for 3 distinct poems.

iterations = ask(
    """Generate three distinct poems in the style of Adrienne Rich's
\"Twenty-One Love Poems\" sequence.

Each poem should feel like it could be a new entry in that sequence —
capturing Rich's directness, her political intimacy, the way she moves
between the personal body and the body politic.

Label them Poem A, Poem B, and Poem C. Make each one different in
tone or focus while staying within the world of the sequence."""
)

print("THREE ITERATIONS")
print("═" * 60)
print(iterations)

---
## Step 3: Identify Shortfalls + Compare Within Context

The group's key insight: the model could *name* Rich's themes but failed to *enact* them. This step asks the model to be a critic — identify what improved from the one-shot, and where the three iterations still fall short.

In [ ]:
# ── Step 3: Compare and identify shortfalls ──────────────────────────────────

comparison = ask(
    f"""Here is a one-shot attempt at an Adrienne Rich poem:

--- ONE-SHOT ---
{one_shot}

--- THREE ITERATIONS (contextualized to \"Twenty-One Love Poems\") ---
{iterations}

Compare these outputs. For each of the three iterations:
1. What specifically improved over the one-shot?
2. What still falls short of Rich's actual voice?
3. Which themes or formal moves does the model *name* but fail to *enact*?

Be specific. Quote lines that work and lines that don't.
End with a list of the 3–5 most important qualities still missing."""
)

print("SHORTFALL ANALYSIS")
print("═" * 60)
print(comparison)

---
## Step 4: Feed in Actual Poems

This is the critical step the group discovered: giving the model the actual text of Rich's poems as ground truth. Instead of relying on the model's training data (which produces generic "Rich-like" output), we provide the specific language.

**Paste 2–3 poems from the *Twenty-One Love Poems* below.** The more specific the source material, the better the output.

In [ ]:
# ── Step 4: Feed in actual poems ─────────────────────────────────────────────
# Paste actual Adrienne Rich poems here.
# These are from "Twenty-One Love Poems" — replace or add more.

ACTUAL_POEMS = [
    {
        "number": "I",
        "text": """Wherever in this city, screens flicker
with pornography, with science-fiction vampires,
victimized hirelings bending to the lash,
we also have to walk...if simply as we walk
through the rainsoaked garbage, the tabloid cruelties
of our own neighborhoods.
We need to grasp our lives inseparable
from those rancid dreams, that blurt of metal, those disgraces,
and the red begonia perilously flashing
from a dark windowsill"""
    },
    {
        "number": "III",
        "text": """Since we're not young, weeks have to do time
for years of missing each other. Yet only this odd warp
in time tells me we're not young.
Did I ever walk the morning streets at twenty,
my limbs streaming with a purer joy?
did I lean from any window over the city
listening for the future
as I listen here with nerves tuned for your ring?"""
    },
    {
        "number": "XII",
        "text": """Every peak is a crater. This is the law of volcanoes,
making them eternally and visibly female.
No height without depth, without a burning core,
though our stale safety counsels on all sides
Hug the fixed shore."""
    }
]

poems_text = "\n\n---\n\n".join(
    f"Poem {p['number']}:\n{p['text']}" for p in ACTUAL_POEMS
)

print(f"✓ Loaded {len(ACTUAL_POEMS)} poems from Twenty-One Love Poems")
for p in ACTUAL_POEMS:
    print(f"  · Poem {p['number']} ({len(p['text'].split())} words)")

---
## Step 5: Generate Improved Poem

Now we combine everything: the shortfall analysis (what's missing), the actual poems (ground truth), and the instruction to match themes and style **without copying lines**. This was the group's starred step — the moment the chain pays off.

In [ ]:
# ── Step 5: Generate improved poem ───────────────────────────────────────────
# Combine: shortfall analysis + actual poems + instruction not to copy.

improved_poem = ask(
    f"""I'm writing a new poem in the style of Adrienne Rich's
\"Twenty-One Love Poems\" sequence.

Here are actual poems from the sequence for you to study:

{poems_text}

Here is an analysis of what previous attempts got wrong:

{comparison}

Now generate a new poem that could be the twenty-second love poem
in Rich's sequence. Requirements:
- Match the themes, style, and formal moves of the actual poems above
- Do NOT directly copy or closely paraphrase specific lines
- Capture Rich's characteristic movement between intimate address and
  political awareness
- Use concrete, specific imagery (not abstract "poetic" language)
- Address the specific shortfalls identified in the analysis above

Write only the poem. No title, no explanation."""
)

print("IMPROVED POEM ★")
print("═" * 60)
print(improved_poem)

---
## Step 6: Cross-Chat Critique

The group's final move: take the improved poem into a **new chat** — a fresh context with no memory of the chain that produced it — and ask for an honest critique. In the workshop they literally opened a new browser tab. Here we simulate it by calling the API with only the poem and no prior context.

This is a powerful technique: the fresh model sees only the output, not the instructions that generated it, so its critique isn't biased by knowing what you were *trying* to do.

In [ ]:
# ── Step 6: Cross-chat critique ──────────────────────────────────────────────
# Fresh context — no memory of the chain. Only the poem.

critique = ask(
    f"""Here is a poem that attempts to imitate Adrienne Rich's style,
specifically her \"Twenty-One Love Poems\" sequence. We're trying to
write what might be her twenty-second love poem.

{improved_poem}

Please critique this poem:
1. How closely does it capture Rich's voice? What specifically works?
2. Where does it fall short or feel artificial?
3. Which lines are strongest? Which are weakest?
4. What specific revisions would make it feel more like Rich?

Be direct and specific. Quote lines in your critique.""",
    system="You are a poetry critic and scholar of Adrienne Rich's work. Be honest and specific in your assessment."
)

print("CROSS-CHAT CRITIQUE")
print("═" * 60)
print(critique)

---
## Step 7: Final Generation

Incorporate the critique. This is the last generation — taking the improved poem and the fresh critique and producing a final version.

In [ ]:
# ── Step 7: Final generation ─────────────────────────────────────────────────
# Back to the original context: actual poems + improved poem + critique.

final_poem = ask(
    f"""Here are actual poems from Adrienne Rich's \"Twenty-One Love Poems\":

{poems_text}

Here is our attempt at a twenty-second love poem:

{improved_poem}

Here is a critique of that attempt:

{critique}

Now write a revised version of the poem that addresses the critique.
Keep what works. Fix what doesn't. Match Rich's voice more closely.
Do not copy her lines — write something new that belongs in the sequence.

Write only the poem. No title, no explanation."""
)

print("FINAL POEM")
print("═" * 60)
print(final_poem)

---
## Compare All Versions

Now look at the progression. The differences between these versions are the point of the exercise — not just "which is better," but *what each step added*.

In [ ]:
# ── Side-by-side comparison ──────────────────────────────────────────────────

print("PROGRESSION")
print("\n" + "═" * 60)
print("1. ONE-SHOT (no context)")
print("═" * 60)
print(one_shot)

print("\n" + "═" * 60)
print("2. IMPROVED (after feeding actual poems + shortfall analysis)")
print("═" * 60)
print(improved_poem)

print("\n" + "═" * 60)
print("3. FINAL (after cross-chat critique)")
print("═" * 60)
print(final_poem)

print("\n" + "═" * 60)
print("\nFor your essay, consider:")
print("  → What did the contextualization step add?")
print("  → What did feeding in actual poems change?")
print("  → What did the cross-chat critique catch?")
print("  → Which step mattered most — and how do you know?")

---
## Going Further

This chain is a starting point. Here are ways to extend it for your assignment:

**Add more poems.** The group used selections from the Twenty-One Love Poems. Try feeding in 5+ poems, or poems from a different Rich collection, and see how the output changes.

**Run the critique loop again.** Take the final poem, critique it, revise it. Repeat 2–3 times. Does quality keep improving, or does it plateau?

**Try the slider approach.** The Cavafy group reframed qualities as sliders (increase intimacy by 20%, decrease abstraction by 30%). You could apply this to Rich's balance between the personal and political.

**Generate love song lyrics instead.** Swap the generation prompt to ask for song lyrics — verse, chorus, bridge — instead of a poem. How does the model handle Rich's voice in a different form?

**Submitting your work:**
- **Lyrics**: Submit an album's worth of songs, with your favorite first
- **Audio**: Take your best lyrics to [Suno](https://suno.com) and generate audio
- **Essay** (500–700 words): Explain your prompt chain, include sample prompts, and reflect on what GPT-4o got right and wrong about your poet